# Optimisation des Coûts de Transport
## Étape 1 : Exploration des Données (Supply Chain)

Dans ce notebook, j'explore le jeu de données pour comprendre comment les produits sont expédiés, quels sont les coûts associés aux différents modes de transport (Aérien, Maritime, Terrestre) et j'identifie les premières pistes d'optimisation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration visuelle
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

data_path = "../data/SCMS_Delivery_History_Dataset.csv"
print("Chargement des données...")
df = pd.read_csv(data_path)

print(f"Dimensions du dataset : {df.shape}")
print("\nAperçu des colonnes :")
df.head(3)

### 1. Nettoyage des données de coût et de poids
Certaines colonnes comme le poids (`Weight (Kilograms)`) ou le coût (`Freight Cost (USD)`) peuvent contenir des caractères non numériques (ex: "See ERP"). Je les nettoie.

In [ ]:
# Nettoyage du poids
df['Weight (Kilograms)'] = pd.to_numeric(df['Weight (Kilograms)'].replace('Weight Captured Separately', np.nan), errors='coerce')

# Nettoyage du coût de transport
df['Freight Cost (USD)'] = pd.to_numeric(df['Freight Cost (USD)'].replace('Freight Included in Commodity Cost', np.nan).replace('Invoiced Separately', np.nan), errors='coerce')

# Nettoyage des dates pour calculer les délais
df['Scheduled Delivery Date'] = pd.to_datetime(df['Scheduled Delivery Date'], errors='coerce')
df['Delivered to Client Date'] = pd.to_datetime(df['Delivered to Client Date'], errors='coerce')
df['PO Sent to Vendor Date'] = pd.to_datetime(df['PO Sent to Vendor Date'], errors='coerce')

# Calcul du délai réel de livraison en jours
df['Delivery_Delay_Days'] = (df['Delivered to Client Date'] - df['PO Sent to Vendor Date']).dt.days

df = df.dropna(subset=['Shipment Mode', 'Weight (Kilograms)', 'Freight Cost (USD)'])
print(f"Dimensions après nettoyage : {df.shape}")

### 2. Analyse des Coûts par Mode de Transport
Je compare le coût total et le coût moyen par expédition selon qu'on utilise l'avion, le bateau ou le camion.

In [ ]:
cost_by_mode = df.groupby('Shipment Mode').agg(
    Total_Cost=('Freight Cost (USD)', 'sum'),
    Avg_Cost=('Freight Cost (USD)', 'mean'),
    Count=('ID', 'count')
).reset_index()

print("Analyse globale par mode de transport :")
print(cost_by_mode.to_string(index=False))

plt.figure(figsize=(10, 5))
sns.barplot(data=cost_by_mode, x='Shipment Mode', y='Total_Cost', palette='viridis')
plt.title('Coût Total de Transport par Mode')
plt.ylabel('Coût (USD)')
plt.show()

### 3. Analyse des Délais par Mode de Transport
Le transport aérien est cher, mais est-il vraiment plus rapide de manière significative pour justifier ce coût ?

In [ ]:
delay_by_mode = df.groupby('Shipment Mode')['Delivery_Delay_Days'].mean().reset_index().dropna()

plt.figure(figsize=(10, 5))
sns.barplot(data=delay_by_mode, x='Shipment Mode', y='Delivery_Delay_Days', palette='magma')
plt.title('Délai Moyen d\'Expédition (Jours) par Mode')
plt.ylabel('Jours')
plt.show()

### 4. Top Routes (Pays de destination) les plus coûteuses
Où l'argent est-il principalement dépensé ?

In [ ]:
top_routes = df.groupby('Country')['Freight Cost (USD)'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_routes.values, y=top_routes.index, palette="Reds_r")
plt.title('Top 10 des Pays Destinataires par Coût de Transport')
plt.xlabel('Coût Total (USD)')
plt.ylabel('Pays')
plt.show()

### 5. Corrélation Poids vs Coût
J'utilise un scatter plot pour voir si le poids impacte linéairement le coût, selon le mode.

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df[df['Weight (Kilograms)'] < 50000], x='Weight (Kilograms)', y='Freight Cost (USD)', hue='Shipment Mode', alpha=0.6)
plt.title("Relation entre Poids et Coût par Mode de Transport (hors valeurs extrêmes)")
plt.show()

## Conclusions Business

1. **Dépendance à l'Aérien (Air)** : Une écrasante majorité des coûts est siphonnée par le fret aérien. S'il est certes plus rapide, la différence de délai avec l'océan ou la route justifie-t-elle la différence colossale de coût ? 
2. **Opportunité d'Optimisation** : En remplaçant une fraction des expéditions aériennes par du maritime (Ocean) ou du routier (Truck) sur des lignes régulières et prévisibles, nous pourrions réaliser d'immenses économies sans casser la supply chain.
3. **Poids et Coûts** : Le graphique de dispersion montre que l'aérien devient exponentiellement cher avec le poids. Imposer un basculement systématique sur l'Océan pour les colis dépassant un certain poids serait une règle métier très rentable.